# 🚧🚧🚧 READ, EXTREMELY IMPORTANT 🚧🚧🚧
**🚧 PLEASE PUT "PINC\dataset\p4gcc\data\p4_ds_training_ready.jsonl" INTO CURRENT DIRECTORY FOR THIS CODE TO RUN!** 🚧

#### Imports and Dependency Resolution
Here, we're installing all the necessary libraries, main of which include:

**transformers:** allows to load pretrained models, tokenizers, configurations

**datasets:** allows to work with the dataset we've produced and saved as .jsonl

**peft:** we use this library to insert LoRA layers

**bitsandbytes:** allows for quantizing the model, which combined with LoRA gives QLoRA





In [1]:
!pip install transformers datasets accelerate wandb peft sentencepiece bitsandbytes
!pip install -U datasets
!pip install tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
import psutil
import torch

from datasets import load_dataset, DatasetDict, Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          PreTrainedTokenizer, PreTrainedModel, TrainingArguments,
                          Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig, PreTrainedTokenizerBase
                          )

from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.set_float32_matmul_precision("high")
import gc
from transformers import DataCollatorWithPadding
import math

#### Enable Quantization Flag

In [3]:
ENABLE_QUANTIZATION_FOR_LORA = True
print(f'QUANTIZATION HAS BEEN {"ENABLED" if ENABLE_QUANTIZATION_FOR_LORA else "DISABLED"}')

QUANTIZATION HAS BEEN ENABLED


#### Utils to Track Memory and Set Seed
**Seed:** the seed is set to 42 to allow for precise reproducibility

In [4]:
def print_mem_usage():
    process = psutil.Process()
    ram_used = process.memory_info().rss / 1000**2

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1000**2
        reserved = torch.cuda.memory_reserved() / 1000**2
        total = torch.cuda.get_device_properties(0).total_memory / 1000**2
        free = reserved - allocated

        print(f"CPU RAM Used       : {ram_used:.2f} MB")
        print(f"GPU VRAM Allocated : {allocated:.2f} MB")
        print(f"GPU VRAM Reserved  : {reserved:.2f} MB")
        print(f"GPU VRAM Free (torch): {free:.2f} MB")
        print(f"GPU VRAM Total     : {total:.2f} MB")
    else:
        print(f"CPU RAM Used       : {ram_used:.2f} MB")
        print("GPU not available.")




In [5]:
import random
import numpy as np
import torch
import os

def set_seed(seed: int = 42):
    random.seed(seed)  # Python RNG
    np.random.seed(seed)  # NumPy RNG
    torch.manual_seed(seed)  # PyTorch CPU RNG
    torch.cuda.manual_seed(seed)  # PyTorch current GPU RNG
    torch.cuda.manual_seed_all(seed)  # All GPUs
    torch.backends.cudnn.deterministic = True  # Makes results deterministic
    torch.backends.cudnn.benchmark = False  # Disables autotuner that could introduce randomness
    os.environ["PYTHONHASHSEED"] = str(seed)  # Python hashing (used in dicts, sets, etc.)


In [6]:
set_seed(42)

#### Utils for Loading Dataset File and Making Train/Val/Split

In [7]:
def load_ds(path: str = "p4_ds_training_ready.jsonl"):
  dataset = load_dataset("json", data_files=path)
  return dataset


In [8]:
def train_val_test_split(dataset: Dataset, train_size: float = 0.85, val_size: float = 0.05, test_size: float = 0.10):
  assert train_size + val_size + test_size == 1.0

  X = dataset.train_test_split(train_size=train_size)

  X2 = X["test"].train_test_split(train_size = (val_size / (1 - train_size)) )

  return DatasetDict(
    {
      "train": X["train"],
      "validation": X2["train"],
      "test": X2["test"]
    }
  )


#### Model Loader Class (TODO: still needs some refinement)
This class allows its user to manage the model instance:

1. That it, it can load the model in a singleton fashion (to avoid loading same model over and over again and polluting the memory by mistake)
2. allows to delete the instance if needed

3. ***NOTE 1:*** uncomment: `attn_implementation = "flash_attention_3"` in `get_instance()` if you have flash attention 3 installed.
4. ***NOTE 2:*** If you're not using Flash Attention 3, feel free to enable attention dropout by commenting out line 19: `config.attention_dropout = 0.0`

In [10]:
# from transformers.utils.import_utils import clear_import_cache

from transformers import AutoConfig

class ModelLoader:
  _instance = {}

  def __init__(self):
    raise RuntimeError("This is a singleton class! Use get_instance(checkpoint: str) method instead!")

  @classmethod
  def get_instance(cls, checkpoint: str = "bigcode/starcoder2-7b"):
    if checkpoint in cls._instance:
      return cls._instance[checkpoint]

    config = AutoConfig.from_pretrained(checkpoint)

    # attention dropout is disabled to work with flash-attention-3. If you're not using it, you can enable it.
    config.attention_dropout = 0.0

    # if you have set the ENABLE_QUANTIZATION_FOR_LORA=False,
    # model would be loaded without this quantization config giving you a LoRA setup, not QLoRA
    if ENABLE_QUANTIZATION_FOR_LORA:
        nf4_config = BitsAndBytesConfig(
          load_in_4bit=True,
          bnb_4bit_quant_type="nf4",
          bnb_4bit_use_double_quant=True,
          bnb_4bit_compute_dtype=torch.bfloat16
        )

    # enable attn_implementation = "flash_attention_3" if you have it installed.
    model = AutoModelForCausalLM.from_pretrained(
      checkpoint,
      quantization_config=nf4_config if ENABLE_QUANTIZATION_FOR_LORA else None,
      torch_dtype=torch.bfloat16,
      trust_remote_code=True,
      device_map="auto",
      # attn_implementation = "flash_attention_3",
      config=config
    )

    model.config.use_cache = False
    model.gradient_checkpointing_disable()

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True)

    cls._instance[checkpoint] = {"model": model, "tokenizer": tokenizer}

    return cls._instance[checkpoint]

  @classmethod
  def delete_instance(cls, checkpoint: str = "bigcode/starcoder2-7b"):
      if checkpoint not in cls._instance:
          return

      instance = cls._instance[checkpoint]

      model = instance.get("model")

      if model is not None:
          model.cpu()
          for attr in dir(model):
              try:
                  delattr(model, attr)
              except:
                  pass
          del model


      tokenizer = instance.get("tokenizer")
      if tokenizer is not None:
          del tokenizer

      del cls._instance[checkpoint]

      gc.collect()
      torch.cuda.empty_cache()


#### Tokenization Functions
This part of the code is what turns raw text into integer indices that are fed into the LLM.

**Few Important Notes:**
1. All of text is surrounded by 2 special tokens: ***\<p4>*** and ***\</p4>***
2. Padding token **[PAD]** is used to make example of different length match in size when passed as a batch
3. Currently maximum size of example fed into the model for training is set to 2048 in the cell below.  This could be increased futher, in which case you will have to reduce LOCAL_BATCH_SIZE in the training loop to avoid CUDA_OUT_OF_MEMORY_ERROR.
4. If any example is longer than 2048, it's converted into multiple examples (i.e. we slide a window across the training example. Managed by following argument that are passed to the call to the tokenizer. Look into stride argument to change the stride of the sliding window.

```

# JUST IN CASE, THIS IS NOT A CELL, DON'T TRY TO RUN IT!

tokenized = tokenizer(..., ..., return_overflowing_tokens=True, truncation=True)
```






In [11]:
MAXIMUM_SEQ_LEN = 2048

In [12]:
def add_special_tokens_to_tokenizer(tokenizer: PreTrainedTokenizerBase, model: AutoModelForCausalLM, new_tokens=["<p4>", "</p4>"]):
  existing = tokenizer.additional_special_tokens
  combined_special_tokens = list(set(existing + new_tokens))
  tokenizer.add_special_tokens({"additional_special_tokens": combined_special_tokens})

  # add special padding token if needed for collation
  if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

  emb_size_mltpl_32 = math.ceil(len(tokenizer) / 32) * 32
  model.resize_token_embeddings(emb_size_mltpl_32)

In [13]:
def tokenize_dataset(dataset, tokenizer):

  def tokenize_ds(examples):
    # the 2048 - 2 is because we are going to add <p4> and </p4>

    tokenized = tokenizer(examples["cleaned_p4"], max_length=MAXIMUM_SEQ_LEN - 2, return_overflowing_tokens=True, truncation=True)
    return tokenized

  def pad_with_p4_tokens(examples):
    p4_start_token_id = tokenizer.encode("<p4>")[0]
    p4_end_token_id = tokenizer.encode("</p4>")[0]

    input_ids = [ [p4_start_token_id] + input_id_list + [p4_end_token_id] for input_id_list in examples["input_ids"]  ]
    attn_mask = [ [1] + attn_mask_list + [1] for attn_mask_list in examples["attention_mask"]  ]

    return {"input_ids": input_ids, "attention_mask": attn_mask}

  dataset = dataset.map(tokenize_ds, batched=True, remove_columns=ds["train"].column_names, num_proc=os.cpu_count())
  dataset = dataset.map(pad_with_p4_tokens, batched=True, remove_columns=["overflow_to_sample_mapping"], num_proc=os.cpu_count() )
  return dataset

#### Data Loader

The dataloader allows to batch the sequences and pad the properly.

In [14]:
def prepare_dataloaders(dataset_splits: DatasetDict, tokenizer, batch_size=8):
  ret = {}

  collate_fn = DataCollatorWithPadding(tokenizer, padding='max_length', max_length=MAXIMUM_SEQ_LEN) #padding=True, 'max_length'

  for split in dataset_splits:
    dataloader = torch.utils.data.DataLoader(
        dataset=dataset_splits[split],
        collate_fn=collate_fn,
        batch_size=batch_size,
        pin_memory=True,
        num_workers=os.cpu_count()
      )

    ret[split] = dataloader

  return ret

#### LoRA

Here, we're applying the lora config with r=32, lora_alpha=64 on q_proj and v_proj with a dropout of 0.05

In [15]:
# training loop without trainer:  https://discuss.huggingface.co/t/training-loop-for-lora/106885

# how to prep model for qlora: https://huggingface.co/docs/peft/en/developer_guides/quantization
# https://huggingface.co/docs/peft/task_guides/prompt_based_methods

def prepare_model_for_qlora(model, lora_config=None):
  if hasattr(model, "peft_config"):
    raise RuntimeError("Model already has peft config, delete it and run again!")

  lora_config = LoraConfig(
    # r=16,
    r = 32,
    # lora_alpha=32,
    lora_alpha = 64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    # lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
  )

  # if you have set the ENABLE_QUANTIZATION_FOR_LORA=False,
  # you would be loading this model in its full size which would make it LoRA, not QLoRA
  if ENABLE_QUANTIZATION_FOR_LORA:
      model = prepare_model_for_kbit_training(model)

  model = get_peft_model(model, lora_config)

  model.print_trainable_parameters()

  return model

#### Make calls to load the data, make splits, load model, prepare it for QLoRA, tokenizer, add special tokens, and tokenize the data

In [16]:

ds = train_val_test_split(load_ds()["train"])

model, tokenizer = ModelLoader.get_instance().values()

add_special_tokens_to_tokenizer(tokenizer, model)

tokenized_dataset_splits = tokenize_dataset(ds, tokenizer)


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/893 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

Map (num_proc=12):   0%|          | 0/89 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/178 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/2419 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/161 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/249 [00:00<?, ? examples/s]

In [18]:
try:
  model = prepare_model_for_qlora(model)
except RuntimeError as e:
  print("LoRA config had already been applied! Deleting model instance. Run the cell above again")
  ModelLoader.delete_instance()


trainable params: 14,680,064 || all params: 7,188,751,360 || trainable%: 0.2042


#### Training Loop


`generate_from_model()` is a utility function to sample model generations during training

In [23]:
def generate_from_model(model, tokenizer, prompt: str, max_new_tokens = 200):
  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

  generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        eos_token_id=tokenizer.convert_tokens_to_ids("</p4>"),
        pad_token_id=tokenizer.convert_tokens_to_ids("[PAD]"),
        repetition_penalty=1.3
  )

  generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
  print(f"\n=== Completion for prompt: {prompt}")
  print(generated_text)
  print("========================================\n")

In [ ]:
# tokenizer.decode([310, 10225, 49153, 49154, 49154, 49154], skip_special_tokens=False)

'\n        poly</p4>[PAD][PAD][PAD]'

`get_gradient_norm()` is a utility function to track the norm of the gradient vector


In [19]:
def get_gradient_norm(model, norm_type=2):
    total_norm = 0.0
    parameters = [p for p in model.parameters() if p.grad is not None]

    if len(parameters) == 0:
        tqdm.write("No gradients found.")
        return

    for p in parameters:
        param_norm = p.grad.data.norm(norm_type)
        total_norm += param_norm.item() ** norm_type

    total_norm = total_norm ** (1. / norm_type)
    return total_norm


`compute_validation_loss(loader: torch.utils.data.Dataloader)` is a utility function compute NLL (Negative Log Likelihood) over a data in a dataloader. You can raise e to power of the returned value to get perplexity:

$$perplexity = e^{compute\_validation\_loss(loader)}$$

In [21]:
def compute_validation_loss(loader):
    model.eval()
    val_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to("cuda") for k, v in batch.items()}
            batch["labels"] = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)

            with autocast("cuda", dtype=torch.bfloat16):
                loss = model(**batch).loss

            val_loss += loss.item()
            num_batches += 1

    return val_loss / num_batches if num_batches > 0 else float("inf")


In [ ]:
    model.eval()
    with torch.no_grad():
      print("MODEL GENERATION BEFORE FINETUNING \n\n")

      generate_from_model(model, tokenizer, "<p4>#include <core.p4>")

#### Finally, the training loop 😸

In [ ]:
import torch
from tqdm import tqdm
from torch.amp import autocast
import time

EPOCHS = 5
FULL_BATCH_SIZE = 64

# set this higher if you have more VRAM (On 96 GiB GPU GH200, 32 works)
# aim for nicer numbers (multiples of 2,4,8,.. and/or powers of 2) to improve training speed
# 8 should work for 40GB VRAM.
LOCAL_BATCH_SIZE = 8
assert FULL_BATCH_SIZE % LOCAL_BATCH_SIZE == 0

split_dataloaders_dict = prepare_dataloaders(tokenized_dataset_splits, tokenizer, LOCAL_BATCH_SIZE)


optimizer = torch.optim.AdamW(model.parameters(), 5e-5, fused=True)

grad_accum_steps_done = 0
grad_accum_loss = 0.0
optimizer_steps = 0

print("MODEL VALID LOSS BEFORE FINE-TUNING:")
model.eval()
with torch.no_grad():
    val_loss = compute_validation_loss(split_dataloaders_dict["validation"])
    print(f"Validation loss before ft: {val_loss}")

for epoch in range(EPOCHS):
    model.train()

    avg_loss_over_batches = 0
    avg_loss_num = 0


    for batch in tqdm(split_dataloaders_dict["train"]):

        # print(batch)
        batch["labels"] = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)
        batch = {k:v.to("cuda") for k,v in batch.items() }
        # print(batch)
        # break

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = (LOCAL_BATCH_SIZE / FULL_BATCH_SIZE) * model(**batch).loss
            loss.backward()

        grad_accum_loss += loss.item()
        grad_accum_steps_done += 1


        if grad_accum_steps_done % (FULL_BATCH_SIZE // LOCAL_BATCH_SIZE) == 0:
          optimizer.step()
          optimizer_steps += 1

          print(f"||∇f_w|| : {get_gradient_norm(model)} ")
          optimizer.zero_grad(set_to_none=True)
          avg_loss_over_batches += grad_accum_loss
          avg_loss_num += 1
          grad_accum_loss = 0.0

    print("\n GENERATING: \n")
    model.eval()
    with torch.no_grad():
        generate_from_model(model, tokenizer, "<p4>#include <core.p4>")
        generate_from_model(model, tokenizer, "def add(x, y")
        val_loss = compute_validation_loss(split_dataloaders_dict["validation"])
        print(f"Validation loss on epoch {epoch + 1}: {val_loss}")

    model.save_pretrained(f"lora_checkpoint_e{epoch}/")
    tokenizer.save_pretrained(f"lora_checkpoint_e{epoch}/")



#### Save the models checkpoint at last epoch, save the tokenizer

In [ ]:
from peft import PeftModel

# Save only LoRA adapter weights
model.save_pretrained("lora_checkpoint/")
tokenizer.save_pretrained("lora_checkpoint/")

/home/ubuntu/.local/lib/python3.10/site-packages/peft/utils/save_and_load.py:252: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('lora_checkpoint_e4/tokenizer_config.json',
 'lora_checkpoint_e4/special_tokens_map.json',
 'lora_checkpoint_e4/vocab.json',
 'lora_checkpoint_e4/merges.txt',
 'lora_checkpoint_e4/added_tokens.json',
 'lora_checkpoint_e4/tokenizer.json')

#### When reusing the code, this is how the model should be loaded:

playing around with `generated_ids = model.generate(...)` arguments can significantly improve generation

In [ ]:
def generate_from_model(model, tokenizer, prompt: str, max_new_tokens = 200):
  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

  generated_ids = model.generate(
        **inputs,
        max_new_tokens=2048,
        eos_token_id=tokenizer.convert_tokens_to_ids("</p4>"),
        pad_token_id=tokenizer.convert_tokens_to_ids("[PAD]"),
        repetition_penalty=1.3,
        top_k = 20,
        top_p=0.8,
        temperature=0.9,
        do_sample=True,
        num_return_sequences = 3
  )

  generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
  print(f"\n=== Completion for prompt: {prompt}")
  print(generated_text)
  print("========================================\n")

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
import torch
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("lora_checkpoint_e4")

# Load quantized base model again
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    "bigcode/starcoder2-7b",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

base_model.resize_token_embeddings(49184)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, "lora_checkpoint_e4")
model.eval()


/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-09 07:14:18.077259: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752045258.086581  139168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752045258.091143  139168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752045258.096552  139168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Starcoder2ForCausalLM(
      (model): Starcoder2Model(
        (embed_tokens): Embedding(49184, 4608)
        (layers): ModuleList(
          (0-31): 32 x Starcoder2DecoderLayer(
            (self_attn): Starcoder2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4608, out_features=4608, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4608, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4608, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
         

In [ ]:
model.eval()
with torch.no_grad():
    generate_from_model(model, tokenizer, "#include <core.p4> <p4>  ")

In [ ]:
# import torch
# import math

# def compute_perplexity(model, dataloader):
#     model.eval()
#     loss_sum = 0.0
#     num_tokens = 0

#     with torch.no_grad():
#         for batch in dataloader:
#             labels = batch["input_ids"].clone()
#             batch["labels"] = labels
#             batch = {k: v.to("cuda") for k, v in batch.items()}
#             outputs = model(**batch)
#             loss = outputs.loss
#             loss_sum += loss.item() * torch.count_nonzero(labels != -100).item()
#             num_tokens += torch.count_nonzero(labels != -100).item()

#     ppl = math.exp(loss_sum / num_tokens)
#     return ppl
